In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DateType,TimestampType, FloatType

from pyspark.sql.functions import to_timestamp, date_format, col

from pyspark.sql.functions import *

catalog_name = "openaq"

### Licenses

In [0]:
df_licenses_silver = spark.table(f"{catalog_name}.bronze.brz_licenses")
display(df_licenses_silver)

In [0]:
df_licenses_silver = df_licenses_silver.dropna(subset=["license_name"])

row_count = df_licenses_silver.count()
print(f"Rows after removing nulls: {row_count}")

In [0]:
df_licenses_silver.groupby("license_id").agg(count("*")).show()

In [0]:
df_licenses_silver.groupby("license_name").agg(count("*")).show()

In [0]:
df_licenses_silver = df_licenses_silver.dropDuplicates(["license_id","license_name"])
display(df_licenses_silver)

In [0]:
df_licenses_silver = df_licenses_silver.withColumn("std_license_name",upper(regexp_replace(col("license_name"),"[^a-zA-Z0-9 ]"," ")))
display(df_licenses_silver)

In [0]:
df_licenses_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.silver.slv_licenses")

### Instruments

In [0]:
df_instruments_silver = spark.table(f"{catalog_name}.bronze.brz_instruments")
display(df_instruments_silver)

In [0]:
df_instruments_silver = df_instruments_silver.dropna(subset=["instrument_id"])

row_count = df_instruments_silver.count()
print(f"Rows after removing nulls: {row_count}")

In [0]:
df_instruments_silver = df_instruments_silver.withColumn("std_manufacturer_name",upper(regexp_replace(col("manufacturer_name"),"[^a-zA-Z0-9\-, ]"," ")))
display(df_instruments_silver)

In [0]:
df_instruments_silver.groupby("instrument_id").agg(count("*")).show()

In [0]:
df_instruments_silver = df_instruments_silver.dropDuplicates(["instrument_id"])
display(df_instruments_silver)

In [0]:
df_instruments_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.silver.slv_instruments")

### Locations

In [0]:
df_locations_silver = spark.table(f"{catalog_name}.bronze.brz_locations")
display(df_locations_silver)

In [0]:
df_locations_silver.filter(col("location_id").isNull()).count()

In [0]:
df_locations_silver = df_locations_silver.dropna(subset=["location_id"])

print(f"Rows after removing nulls: {df_locations_silver.count()}")

In [0]:
df_locations_silver.columns

In [0]:
location_columns = ['location_name','locality','country_name','owner_name','provider_name','country_code','instrument_name','license_name','license_attribution_name']

for cols in location_columns:
    df_locations_silver = df_locations_silver.withColumn(f"std_{cols}",upper(regexp_replace(col(f"{cols}"),"[^a-zA-Z0-9 ]"," ")))

display(df_locations_silver)

In [0]:
df_locations_silver = df_locations_silver.withColumn("std_timezone",upper(regexp_replace(col("timezone"),"[^a-zA-Z0-9\/ ]"," ")))

In [0]:
datetime = ['first_seen_utc','last_seen_utc']

for cols in datetime:
    df_locations_silver = df_locations_silver.withColumn(f"std_{cols}",date_format(to_timestamp(cols, "yyyy-MM-dd'T'HH:mm:ssXXX"),"yyyy-MM-dd HH:mm:ss"))

display(df_locations_silver)

In [0]:
df_locations_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.silver.slv_locations")